# Combined Evaluation and System Demonstration

**Owner:** Member 4 (integration, evaluation)
**Purpose:** Evaluate the three models together (face, voice, product) and test the multimodal decision logic used by `src/app.py`, including the four required scenarios.
**Inputs:** `models/*.joblib`, `outputs/metrics/*.json`, `data/processed/*.csv`
**Outputs:** the evaluation tables and the four scenario results below.

The decision logic is the one in the command-line app: face check, hidden recommendation, voice check against the **same** member, and the recommendation shown only if both checks pass.

In [1]:
import os, sys, json, warnings, random
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
random.seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)

if Path.cwd().name == "notebooks":
    os.chdir("..")
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import app
from preprocess_images import _verify_notebook_bundle
from train_auth_models import verify_features
print("Repository root folder:", ROOT.name)

Repository root folder: Formative-2-machine-learning


## 1. Model evaluation table
Metrics come from each model owner's saved metrics file, all measured on held-out data:
- **Face model** (Member 2): leave-one-expression-out cross-validation.
- **Voice model** (Member 3 / Part 4 code): grouped hold-out split by original recording.
- **Product model** (Member 1): hold-out split grouped by customer, so no customer is in both train and test.

In [2]:
M = ROOT / "outputs" / "metrics"
face = json.load(open(M / "face_model_metrics.json"))
voice = json.load(open(M / "voice_metrics.json"))
prod = json.load(open(M / "product_model_metrics.json"))
pm = prod["grouped_split"]["metrics"][prod["best_model"]]
fc = face["cross_validation"]

table = pd.DataFrame([
    {"model": "Face recognition", "classifier": "Logistic Regression on FaceNet embeddings",
     "accuracy": fc["accuracy"], "F1 (weighted)": fc["weighted_f1"], "log loss": fc["log_loss"],
     "evaluated on": f"{fc['authorized_originals_evaluated']} original photos (3-fold)"},
    {"model": "Voiceprint verification", "classifier": voice["model_type"],
     "accuracy": voice["accuracy"], "F1 (weighted)": voice["f1_weighted"], "log loss": voice["log_loss"],
     "evaluated on": f"{voice['test_rows']} held-out rows"},
    {"model": "Product recommendation", "classifier": prod["best_model"],
     "accuracy": pm["accuracy"], "F1 (weighted)": pm["f1_weighted"], "log loss": pm["log_loss"],
     "evaluated on": f"{pm['n_rows']} held-out rows (customer-grouped)"},
]).round(3)
table

,model,classifier,accuracy,F1 (weighted),log loss,evaluated on
0,Face recognition,Logistic Regression on FaceNet embeddings,1.000,1.000,0.006,12 original photos (3-fold)
1,Voiceprint verification,logistic_regression,0.812,0.803,0.551,16 held-out rows
2,Product recommendation,logistic_regression,0.167,0.125,2.751,30 held-out rows (customer-grouped)


**Reading the table.** The face model separates the four members perfectly, but on very few photos. The voice model is clearly weaker. The product model scores **below the 0.20 chance level** for five categories on the customer-grouped test, so its recommendations are only illustrative: the customer features carry little signal for the product category in this small dataset (Member 1 documents the same limitation in the product metrics file).

## 2. Authentication error rates
- **FAR** (false acceptance rate): how often an unauthorized person is accepted.
- **FRR** (false rejection rate): how often an authorized person is rejected.

In [3]:
t = face["threshold_validation"]; f_final = face["final_unauthorized_test"]
face_rates = {"validation FAR": t["false_acceptance_rate"], "validation FRR": t["false_rejection_rate"],
              "final unauthorized test FAR": f_final["false_acceptance_rate"]}

aud = pd.read_csv("data/processed/audio_features.csv")
unauth = pd.read_csv("data/processed/audio_features_unauthorized.csv")
vb = joblib.load("models/voice_model.joblib")
feat = vb["features"]

def voice_decision(row, expected):
    member, conf, ok = verify_features("models/voice_model.joblib", {k: row[k] for k in feat}, expected)
    return {"member": member, "confidence": conf, "accepted": ok}

# FAR: every unauthorized recording is tried against every member it could claim to be
far_trials = [(i, m) for i in range(len(unauth)) for m in sorted(aud.member_id.unique())]
far_accept = sum(voice_decision(unauth.iloc[i], m)["accepted"] for i, m in far_trials)
# FRR: each member's original recordings against their own identity
orig = aud[aud.augmentation == "original"]
frr_reject = sum(not voice_decision(r, r.member_id)["accepted"] for _, r in orig.iterrows())
# Mismatch: a member's recording claimed as a different member
mis = [(r, m) for _, r in orig.iterrows() for m in sorted(aud.member_id.unique()) if m != r.member_id]
mis_accept = sum(voice_decision(r, m)["accepted"] for r, m in mis)

# Honest FRR: the saved model was refitted on ALL recordings, so its own recordings are not a fair test.
# Retrain without one original recording per member (plus that recording's augmented copies), using the
# same threshold and distance limit as the saved model, and test on the held-out recordings.
import tempfile, contextlib, io
from types import SimpleNamespace
import train_auth_models as tam

Zall = vb["model"].named_steps["prep"].transform(aud[feat])
ratios = []
for m in vb["limits"]:
    dist = [np.linalg.norm(z - vb["centroids"][m]) for z, lab in zip(Zall, aud.member_id) if lab == m]
    ratios.append(vb["limits"][m] / max(dist))
DISTANCE_FACTOR = round(float(np.median(ratios)), 2)
print("distance factor of the saved voice model:", DISTANCE_FACTOR, "| confidence threshold:", vb["threshold"])

def retrain(df):
    with tempfile.TemporaryDirectory() as tmp:
        csv = Path(tmp) / "k.csv"; df.to_csv(csv, index=False)
        old_dir = os.getcwd(); os.chdir(tmp)
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                tam.train("voice", SimpleNamespace(data=str(csv), label=None, group_col=None,
                                                   threshold=vb["threshold"], distance_factor=DISTANCE_FACTOR))
            return joblib.load(Path(tmp) / "models" / "voice_model.joblib")
        finally:
            os.chdir(old_dir)

held_total = held_rejected = 0
for k in range(2):
    hold = [sorted(aud[aud.member_id == m].source_file.unique())[k] for m in sorted(aud.member_id.unique())]
    test_rows = aud[aud.source_file.isin(hold)]
    fold_model = retrain(aud[~aud.source_file.isin(hold)])
    outcomes = tam.predict_rows(fold_model, test_rows)
    held_total += len(test_rows)
    held_rejected += sum((g != m) or (not ok) for (g, _, ok), m in zip(outcomes, test_rows.member_id))
print(f"Members' new (held-out) recordings rejected: {held_rejected}/{held_total}")

rates = pd.DataFrame([
    {"model": "Face", "check": "unknown face accepted (FAR)", "result": f"{face_rates['final unauthorized test FAR']:.0%}", "trials": f_final["images_evaluated"]},
    {"model": "Face", "check": "member rejected (FRR, validation)", "result": f"{face_rates['validation FRR']:.0%}", "trials": t["authorized_samples"]},
    {"model": "Voice", "check": "unknown voice accepted (FAR)", "result": f"{far_accept/len(far_trials):.0%}", "trials": len(far_trials)},
    {"model": "Voice", "check": "member rejected (FRR, new held-out recordings)", "result": f"{held_rejected/held_total:.0%}", "trials": held_total},
    {"model": "Voice", "check": "member rejected (training recordings, optimistic)", "result": f"{frr_reject/len(orig):.0%}", "trials": len(orig)},
    {"model": "Voice", "check": "wrong member's voice accepted (mismatch)", "result": f"{mis_accept/len(mis):.0%}", "trials": len(mis)},
])
rates

distance factor of the saved voice model: 1.0 | confidence threshold: 0.6


Members' new (held-out) recordings rejected: 30/32


,model,check,result,trials
0,Face,unknown face accepted (FAR),0%,3
1,Face,"member rejected (FRR, validation)",0%,12
2,Voice,unknown voice accepted (FAR),0%,8
3,Voice,"member rejected (FRR, new held-out recordings)",94%,32
4,Voice,"member rejected (training recordings, optimistic)",0%,8
5,Voice,wrong member's voice accepted (mismatch),0%,24


**Reading the rates.** The face model rejects the unauthorized photos and accepts the members. The saved voice model uses a very strict distance limit, so it accepts no stranger (FAR 0%), but it also **rejects most new recordings of the real members** (high FRR on held-out recordings), so a real member could be denied. The FAR and FRR trade off against each other. Only two original recordings per member exist, so neither can be tuned reliably; more recordings per member are needed. The "training recordings" FRR is shown only to illustrate how misleading a test on training data is.
The unauthorized recordings were used **only to report** these numbers, not to choose thresholds.

## 3. System demonstration: the four scenarios
`run_transaction` in `src/app.py` is the exact decision flow of the command-line app. The raw photos and recordings are biometric data and are not in the repository, so the checks here use the **stored feature rows** (FaceNet embeddings and audio features) through the same saved models and thresholds. With the raw files, the same scenarios run with:

```
python src/app.py --face <photo.jpg> --voice <recording.wav>
```

In [4]:
img = pd.read_csv("data/processed/image_features.csv")
emb_cols = [c for c in img.columns if c.startswith("embedding_")]
face_bundle = joblib.load("models/face_recognition_bundle.joblib")

def face_from_embedding(embedding, name):
    r = _verify_notebook_bundle(face_bundle, np.asarray(embedding, float), name)
    return {"member": r.predicted_member, "confidence": r.classifier_confidence, "accepted": r.accepted}

member_photo = img[(img.member_id == "member01") & (img.augmentation == "original")].iloc[0]
rng = np.random.default_rng(RANDOM_STATE)
stranger = rng.normal(size=512); stranger /= np.linalg.norm(stranger)   # a face unlike any member

voice_member01 = orig[orig.member_id == "member01"].iloc[0]
voice_member02 = orig[orig.member_id == "member02"].iloc[0]
voice_stranger = unauth.iloc[0]

def scenario(title, embedding, voice_row):
    print("#" * 70); print(title); print("#" * 70)
    return app.run_transaction(
        "face", "voice",
        face_checker=lambda _: face_from_embedding(embedding, "photo"),
        voice_checker=lambda _, expected: voice_decision(voice_row, expected))

results = {}
results["1 authorized member"] = scenario("1. Authorized face + the same member's voice", member_photo[emb_cols], voice_member01)
print()
results["2 unauthorized face"] = scenario("2. Unauthorized face", stranger, voice_member01)
print()
results["3 unauthorized voice"] = scenario("3. Authorized face + unauthorized voice", member_photo[emb_cols], voice_stranger)
print()
results["4 voice mismatch"] = scenario("4. Authorized face (member01) + another member's voice (member02)", member_photo[emb_cols], voice_member02)

######################################################################
1. Authorized face + the same member's voice
######################################################################
=== Multimodal authentication and product recommendation ===
Face recognised as member01 (confidence 1.00).


A recommendation is ready. Voice verification is required to display it.
Voice verified as member01 (confidence 0.99).
ACCESS GRANTED. Recommended products:
  1. Books (31%)
  2. Groceries (24%)
  3. Sports (17%)

######################################################################
2. Unauthorized face
######################################################################
=== Multimodal authentication and product recommendation ===
ACCESS DENIED: face not recognised (closest match member01, confidence 0.51).

######################################################################
3. Authorized face + unauthorized voice
######################################################################
=== Multimodal authentication and product recommendation ===
Face recognised as member01 (confidence 1.00).
A recommendation is ready. Voice verification is required to display it.
ACCESS DENIED: voice sounds like member02, not member01 (confidence 0.83).

############################################

In [5]:
pd.DataFrame([{"scenario": k, "outcome": v["outcome"], "reason": v["reason"]} for k, v in results.items()])

,scenario,outcome,reason
0,1 authorized member,GRANTED,face and voice verified
1,2 unauthorized face,DENIED,face not recognised
2,3 unauthorized voice,DENIED,"voice sounds like member02, not member01"
3,4 voice mismatch,DENIED,"voice sounds like member02, not member01"


**Reading the scenarios.** Scenario 1 is granted and shows the recommendation. Scenarios 2 and 4 are denied, and a recommendation is never shown when either check fails. Scenario 3 is denied because the stranger's voice does not match member01. Scenarios 1 and 4 use recordings the voice model was trained on, so scenario 1 passing is optimistic: with a brand-new recording a real member could be rejected (see the held-out FRR above).

## 4. Findings, limitations and next steps
- The multimodal logic works: the recommendation is computed after the face check, kept hidden, and displayed only after the voice matches the **same** member.
- **Face:** strong separation, but only 3 original photos per member and one unknown person were available.
- **Voice:** the weakest link (only 2 original recordings per member). The current model rejects strangers but also most new recordings of real members. Member 3 is improving unknown-voice rejection with separate validation and test data. When the final voice model replaces `models/voice_model.joblib`, re-run this notebook to refresh sections 1 to 3.
- **Product:** below chance on a customer-grouped test, so recommendations should not be treated as reliable.
- **Demo assumption:** there is no member-to-customer table, so each member is linked to one customer from the merged data (member01 to the first customer id, and so on).
- **Privacy:** raw photos and recordings are biometric data and need participant consent; they are kept out of the repository.